# Week 2 Assignment – Core ML Pipeline & First Models
**Coderaxo AI/ML Internship**

This notebook covers:
- Task 1: ML Pipeline Theory
- Task 2: Linear Regression (House Prices dataset)
- Task 3: Logistic Regression (Titanic dataset)
- Task 4: Decision Tree (Titanic dataset)
- Task 5: Preprocessing Pipeline (Pipeline + ColumnTransformer)
- Optional Challenge: Actual vs. Predicted plot


In [ ]:
# Common imports used throughout the notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
RANDOM_STATE = 42


---
## Task 1 – Machine Learning Pipeline (Theory)

A typical ML pipeline has **6 steps**. Below is each step explained in plain terms, followed by
a worked example: **Loan Prediction** (predicting whether a bank should approve a loan application).

1. **Problem Definition** – Decide exactly what you are predicting and why it matters.
   *Example:* We define the goal as predicting `Loan_Status` (Approved / Rejected) for a new
   applicant, so the bank can automate its initial screening decision.

2. **Data Collection** – Gather the raw data needed to solve the problem, from databases,
   files, APIs, or forms.
   *Example:* We collect historical loan applications with fields like income, credit history,
   loan amount, and the final approval outcome.

3. **Data Preprocessing / Cleaning** – Handle missing values, fix inconsistent formats, remove
   duplicates, encode categorical variables, and scale numeric ones so the data is model-ready.
   *Example:* We fill missing `LoanAmount` values with the median, encode `Gender` and
   `Married` as numbers, and scale `ApplicantIncome`.

4. **Exploratory Data Analysis (EDA) / Feature Engineering** – Explore relationships in the
   data and create or select the features that will actually help the model learn.
   *Example:* We notice `Credit_History` strongly correlates with approval, and we engineer a
   `Total_Income = ApplicantIncome + CoapplicantIncome` feature.

5. **Model Training** – Split the data into training and testing sets, choose an algorithm, and
   fit it on the training data.
   *Example:* We train a Logistic Regression model on 80% of the applications to learn the
   pattern between applicant features and loan approval.

6. **Model Evaluation & Deployment** – Measure the model's performance on unseen data using
   appropriate metrics, then deploy it so it can make predictions on new applicants.
   *Example:* We evaluate the model using Accuracy and F1 Score on the remaining 20% of data,
   and if performance is acceptable, deploy it as an API the bank's loan system can call.


---
## Task 2 – Linear Regression
**Dataset:** House Prices – Advanced Regression Techniques (Kaggle)

### 2.1 Load and Explore the Dataset


In [ ]:
house_df = pd.read_csv("house_train.csv")
print("Shape:", house_df.shape)
house_df.head()


In [ ]:
house_df.info()


In [ ]:
house_df.describe()


In [ ]:
# Check missing values (top 15 columns with most missing values)
missing = house_df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(f"Number of columns with missing values: {len(missing)}")
missing.head(15)


### 2.2 Data Preprocessing

For this Linear Regression baseline we:
- Drop the `Id` column (not predictive).
- Select the **numeric** features only, since Linear Regression requires numeric input and this
  keeps the preprocessing simple and interpretable for a first model.
- Impute missing numeric values (e.g. `LotFrontage`, `MasVnrArea`, `GarageYrBlt`) with the
  **median** of each column, since median is robust to outliers (common in price/area data).


In [ ]:
# Drop Id, separate target
house_df = house_df.drop(columns=["Id"])
target = "SalePrice"

# Keep numeric features only for this baseline model
numeric_house_df = house_df.select_dtypes(include=[np.number]).copy()
print("Numeric feature columns:", numeric_house_df.shape[1] - 1)

X_house = numeric_house_df.drop(columns=[target])
y_house = numeric_house_df[target]

# Impute missing numeric values with the median
imputer = SimpleImputer(strategy="median")
X_house_imputed = pd.DataFrame(
    imputer.fit_transform(X_house), columns=X_house.columns, index=X_house.index
)

print("Remaining missing values after imputation:", X_house_imputed.isnull().sum().sum())


### 2.3 Train/Test Split

In [ ]:
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_house_imputed, y_house, test_size=0.2, random_state=RANDOM_STATE
)
print("Train shape:", X_train_h.shape, " Test shape:", X_test_h.shape)


### 2.4 Train Linear Regression Model

In [ ]:
lin_reg = LinearRegression()
lin_reg.fit(X_train_h, y_train_h)

y_pred_h = lin_reg.predict(X_test_h)
print("Model trained.")


### 2.5 Evaluate the Model

In [ ]:
mae = mean_absolute_error(y_test_h, y_pred_h)
mse = mean_squared_error(y_test_h, y_pred_h)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_h, y_pred_h)

print(f"MAE  : {mae:,.2f}")
print(f"MSE  : {mse:,.2f}")
print(f"RMSE : {rmse:,.2f}")
print(f"R2   : {r2:.4f}")


### 2.6 Explanation of Results

The R² score tells us what proportion of the variance in `SalePrice` is explained by our
numeric features — a value closer to 1 means the model fits well. RMSE and MAE are both in
dollar terms, so RMSE (which penalizes larger errors more heavily) being noticeably higher than
MAE indicates the model struggles more on a subset of expensive/unusual houses (outliers). Since
we only used numeric features and skipped categorical information like `Neighborhood` or
`ExterQual`, there's clear room to improve the model by encoding those categorical signals.


---
## Task 3 – Logistic Regression
**Dataset:** Titanic – Machine Learning from Disaster (Kaggle)

### 3.1 Load and Explore the Dataset


In [ ]:
titanic_df = pd.read_csv("titanic_train.csv")
print("Shape:", titanic_df.shape)
titanic_df.head()


In [ ]:
titanic_df.info()


In [ ]:
titanic_df.isnull().sum()


### 3.2 Handle Missing Values

- `Age` (177 missing): impute with the **median** age.
- `Embarked` (2 missing): impute with the **mode** (most frequent port).
- `Cabin` (687 missing, ~77% of the data): too sparse to impute meaningfully, so we drop the
  column entirely rather than introduce noise.


In [ ]:
titanic_clean = titanic_df.copy()

titanic_clean["Age"] = titanic_clean["Age"].fillna(titanic_clean["Age"].median())
titanic_clean["Embarked"] = titanic_clean["Embarked"].fillna(titanic_clean["Embarked"].mode()[0])
titanic_clean = titanic_clean.drop(columns=["Cabin"])

# Drop identifier / free-text columns not useful for a baseline model
titanic_clean = titanic_clean.drop(columns=["PassengerId", "Name", "Ticket"])

print("Missing values remaining:\n", titanic_clean.isnull().sum())


### 3.3 Encode Categorical Features

`Sex` and `Embarked` are converted to numeric form using one-hot encoding (`drop_first=True`
to avoid redundant/multicollinear columns).


In [ ]:
titanic_encoded = pd.get_dummies(titanic_clean, columns=["Sex", "Embarked"], drop_first=True)
titanic_encoded.head()


### 3.4 Scale Numerical Features

In [ ]:
X_titanic = titanic_encoded.drop(columns=["Survived"])
y_titanic = titanic_encoded["Survived"]

num_cols = ["Age", "Fare", "SibSp", "Parch"]
scaler = StandardScaler()
X_titanic_scaled = X_titanic.copy()
X_titanic_scaled[num_cols] = scaler.fit_transform(X_titanic[num_cols])

X_titanic_scaled.head()


### 3.5 Train/Test Split & Train Logistic Regression

In [ ]:
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_titanic_scaled, y_titanic, test_size=0.2, random_state=RANDOM_STATE, stratify=y_titanic
)

log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
log_reg.fit(X_train_t, y_train_t)

y_pred_log = log_reg.predict(X_test_t)
print("Model trained.")


### 3.6 Evaluate the Model

In [ ]:
acc_log = accuracy_score(y_test_t, y_pred_log)
prec_log = precision_score(y_test_t, y_pred_log)
rec_log = recall_score(y_test_t, y_pred_log)
f1_log = f1_score(y_test_t, y_pred_log)

print(f"Accuracy  : {acc_log:.4f}")
print(f"Precision : {prec_log:.4f}")
print(f"Recall    : {rec_log:.4f}")
print(f"F1 Score  : {f1_log:.4f}")


In [ ]:
cm_log = confusion_matrix(y_test_t, y_pred_log)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_log, display_labels=["Did Not Survive", "Survived"])
disp.plot(cmap="Blues")
plt.title("Logistic Regression - Confusion Matrix")
plt.show()


### 3.7 Findings

The confusion matrix shows how many passengers were correctly vs. incorrectly classified for
each class. Precision tells us how many of the passengers we *predicted* as survivors actually
survived, while recall tells us how many of the *actual* survivors we correctly identified. On
the Titanic dataset, `Sex` and `Pclass` are typically the strongest predictors — women and
first-class passengers had a much higher survival rate — which is reflected in the model's
learned coefficients.


---
## Task 4 – Decision Tree
**Dataset:** Titanic – Machine Learning from Disaster (Kaggle)

### 4.1 Train a Decision Tree Classifier
Using the **same preprocessed dataset** (`X_train_t`, `X_test_t`, `y_train_t`, `y_test_t`) from Task 3.


In [ ]:
dt_clf = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)
dt_clf.fit(X_train_t, y_train_t)

y_pred_dt = dt_clf.predict(X_test_t)
acc_dt = accuracy_score(y_test_t, y_pred_dt)
print(f"Decision Tree Accuracy: {acc_dt:.4f}")


### 4.2 Compare with Logistic Regression

In [ ]:
comparison_df = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree"],
    "Accuracy": [acc_log, acc_dt]
})
comparison_df


In [ ]:
plt.bar(comparison_df["Model"], comparison_df["Accuracy"], color=["#4C72B0", "#DD8452"])
plt.ylabel("Accuracy")
plt.title("Logistic Regression vs. Decision Tree")
plt.ylim(0, 1)
for i, v in enumerate(comparison_df["Accuracy"]):
    plt.text(i, v + 0.02, f"{v:.3f}", ha="center")
plt.show()


### 4.3 Which Model Performed Better and Why

Comparing the two accuracy scores above: whichever model scores higher does so because of how
it handles the relationships in the data. Logistic Regression works best when the relationship
between features and the outcome is roughly linear (e.g., higher fare or first class steadily
increasing survival odds), and it tends to generalize well on smaller, cleaner datasets. A
Decision Tree can capture more complex, non-linear interactions (e.g., "if male AND Pclass=3
AND Age>10, survival drops sharply") but with `max_depth=5` it is constrained to avoid
overfitting — an unconstrained tree would likely memorize the training data and perform worse
on the test set. In practice, on Titanic-sized data (~900 rows) the two models often perform
similarly, with the better one depending on the exact train/test split and hyperparameters.


---
## Task 5 – Preprocessing Pipeline
**Dataset:** Titanic – Machine Learning from Disaster (Kaggle)

### 5.1 Build a Preprocessing Workflow with `Pipeline` and `ColumnTransformer`

This time we build the preprocessing as a single reusable pipeline object, starting from the
**raw** Titanic data (before manual cleaning), so imputation/encoding/scaling all happen inside
the pipeline itself.


In [ ]:
# Start from raw data again, only dropping columns that are pure identifiers/free text
titanic_raw = titanic_df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

X_raw = titanic_raw.drop(columns=["Survived"])
y_raw = titanic_raw["Survived"]

numeric_features = ["Age", "Fare", "SibSp", "Parch"]
categorical_features = ["Sex", "Embarked", "Pclass"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])


### 5.2 Train a Classification Model Using the Pipeline

In [ ]:
full_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=RANDOM_STATE, stratify=y_raw
)

full_pipeline.fit(X_train_p, y_train_p)
y_pred_pipeline = full_pipeline.predict(X_test_p)

acc_pipeline = accuracy_score(y_test_p, y_pred_pipeline)
print(f"Pipeline Model Accuracy: {acc_pipeline:.4f}")


### 5.3 Explanation

**Why Pipelines are useful:**
A `Pipeline` chains every preprocessing step and the final model into a single object. This
means we call `.fit()` and `.predict()` once, the exact same transformations are guaranteed to
be applied consistently to training and test data, and the entire workflow can be saved,
reused, or dropped into cross-validation as one unit — instead of manually repeating
imputation/encoding/scaling code and risking mistakes.

**What Data Leakage is:**
Data leakage happens when information from outside the training set (often from the test set,
or from the target itself) accidentally influences model training — making the model look more
accurate than it really is. A common example: imputing missing values or fitting a scaler using
statistics (mean, median) computed from the **entire** dataset (train + test combined) before
splitting — the model then indirectly "sees" information about the test set during training.

**How Pipelines help prevent Data Leakage:**
When a `Pipeline` is used inside `train_test_split` + `.fit()`, all the `fit_transform` steps
(imputer, scaler, encoder) are fit **only on the training fold**, and only `.transform()` is
applied to the test fold. This is enforced automatically, including inside cross-validation,
so statistics from the test set can never leak into the preprocessing of the training set.


---
## Optional Challenge – Actual vs. Predicted Plot (Linear Regression)


In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test_h, y_pred_h, alpha=0.5, color="#4C72B0")
min_val = min(y_test_h.min(), y_pred_h.min())
max_val = max(y_test_h.max(), y_pred_h.max())
plt.plot([min_val, max_val], [min_val, max_val], color="red", linestyle="--", label="Perfect Prediction")
plt.xlabel("Actual SalePrice")
plt.ylabel("Predicted SalePrice")
plt.title("Actual vs. Predicted House Prices (Linear Regression)")
plt.legend()
plt.tight_layout()
plt.show()


**Explanation:** Each point represents one house in the test set, plotted by its actual
sale price (x-axis) against the model's predicted price (y-axis). Points falling exactly on the
red dashed line would mean a perfect prediction; the tighter the cloud of points hugs that line,
the better the model. Points that drift further from the line — usually at the higher price
end — show where the model's numeric-only features aren't capturing enough information to
price expensive homes accurately.
